<a href="https://colab.research.google.com/github/e22340-SajithaSanthushti/Statistical-Learning-e22340/blob/main/Bayesian_Inference_AssignmentE22340.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate a range for theta (latent ability)
theta_vals = np.linspace(-4, 4, 400)

# Configurations: Two distinct values of a_i (e.g., 0.8 and 1.8)
# One of those a_i values (1.8) is paired with three different b_i values (-1, 0, 1)
curves = [
    {"a": 0.8, "b": 0, "name": "a = 0.8, b = 0 (Low Discrimination)"},
    {"a": 1.8, "b": -1, "name": "a = 1.8, b = -1 (High Disc, Easy)"},
    {"a": 1.8, "b": 0, "name": "a = 1.8, b = 0 (High Disc, Medium)"},
    {"a": 1.8, "b": 1, "name": "a = 1.8, b = 1 (High Disc, Hard)"},
]

# Create the Plotly figure
fig = go.Figure()

for c in curves:
    y_vals = p_i(theta_vals, c["a"], c["b"])
    fig.add_trace(go.Scatter(x=theta_vals, y=y_vals, mode='lines', name=c["name"]))

# Update layout styling
fig.update_layout(
    title="2PL Item Response Category Curves (Visualizing $a_i$ and $b_i$)",
    xaxis_title="User Latent Ability ($\\theta$)",
    yaxis_title="Probability of Correct Response $P(Y_i = 1 | \\Theta = \\theta)$",
    template="plotly_white",
    hovermode="x unified"
)

# Show plot
fig.show()

1. Likelihood Contribution of a Single ResponseThe response $y_k \in \{0, 1\}$ at step $k$ follows a Bernoulli distribution governed by the probability $p_k(\theta)$. We can write the single-item likelihood contribution $L(y_k \mid \theta)$ compactly using the exponent form:$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$Substituting the 2PL logistic function $p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$, this expands to:$$L(y_k \mid \theta) = \left( \frac{1}{1 + e^{-a_k(\theta - b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}} \right)^{1 - y_k}$$


Using Bayes' theorem, the posterior density at step $k$ is proportional to the product of the likelihood contribution of the new observation $y_k$ and the prior distribution at step $k$ (which is the posterior distribution from step $k-1$).The recursive relationship up to a proportionality constant is:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Substituting the Bernoulli likelihood contribution for $L(y_k \mid \theta)$, the relation is written as:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

A correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) shifts the peak of the running posterior density distribution to the right (toward a higher ability estimate $\theta$) relative to the previous step.Mathematically, this shift can be understood by analyzing the mode of the posterior (the Maximum A Posteriori estimate) via the log-posterior derivative:1. The Gradient ContributionThe log-posterior density at step $k$ is the sum of the log-likelihood and the previous log-posterior:$$\log f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \log p_k(\theta) + \log f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) + C$$Taking the derivative with respect to $\theta$ gives the updating gradient:$$\frac{d}{d\theta} \log f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{d}{d\theta} \log p_k(\theta) + \frac{d}{d\theta} \log f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$For the 2PL logistic model, the derivative of the log-likelihood for a correct response ($y_k = 1$) is:$$\frac{d}{d\theta} \log p_k(\theta) = a_k \big(1 - p_k(\theta)\big)$$2. The Impact of a Large $b_k$Let $\theta_{k-1}$ be the peak (mode) of the previous distribution, where the old gradient is zero: $\left. \frac{d}{d\theta} \log f_{\Theta \mid \mathbf{Y}^{(k-1)}} \right\vert{}_{\theta_{k-1}} = 0$.Evaluating the new gradient at this old peak yields:$$\left. \frac{d}{d\theta} \log f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \right\vert{}_{\theta = \theta_{k-1}} = a_k \big(1 - p_k(\theta_{k-1})\big)$$Because the item is highly difficult ($b_k \gg \theta_{k-1}$), the term $(\theta_{k-1} - b_k)$ is a large negative number.This makes the probability of a correct response at the current estimated ability very low: $p_k(\theta_{k-1}) \approx 0$.As a result, the term $\big(1 - p_k(\theta_{k-1})\big)$ approaches its upper bound of $1$.

The discrimination parameter $a_k$ controls how much information the item provides about the user's latent ability $\theta$. In a Bayesian framework, more information translates directly to an increase in precision, which decreases the variance and sharpens the posterior distribution.Mathematically, we can see this by looking at the Fisher Information or the second derivative of the log-posterior (which approximates the inverse variance at the peak via a Laplace approximation). The second derivative of the log-likelihood for a response $y_k$ is:$$\frac{d^2}{d\theta^2} \log L(y_k \mid \theta) = -a_k^2 \, p_k(\theta)\big(1 - p_k(\theta)\big)$$The negative curvature of the posterior density at step $k$ accumulates this contribution:$$-\frac{d^2}{d\theta^2} \log f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = -\frac{d^2}{d\theta^2} \log f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) + a_k^2 \, p_k(\theta)\big(1 - p_k(\theta)\big)$$Because the variance is approximately the inverse of this curvature, the magnitude of $a_k$ dictates the change in sharpness.When $a_k$ is Very Large (High Discrimination)Mathematical effect: The value of $a_k^2$ becomes massive, significantly increasing the negative curvature at the peak of the log-posterior.Impact on the distribution: The variance drops sharply. The posterior distribution becomes highly concentrated and narrow ("sharp") around the updated estimate. This signifies that the platform has gained a high level of certainty about the user's true ability from this single item because the question is exceptionally good at separating ability levels.When $a_k$ is Very Small (Low Discrimination)Mathematical effect: As $a_k \to 0$, the update term $a_k^2 \, p_k(\theta)\big(1 - p_k(\theta)\big)$ approaches zero.Impact on the distribution: The curvature barely changes, meaning the variance remains almost identical to the previous step. The posterior distribution retains its width and flat appearance, experiencing negligible sharpening. This indicates a low-certainty update because a low-discrimination question is noisy and provides almost no new diagnostic information to refine the platform's estimate.

An algorithmic approach to maintaining the running posterior density on a fixed grid of $\theta$-values (often called a grid approximation or quadrature method) can be broken down into initialization, sequential updating, and numerical normalization.1. Grid Definition and InitializationDefine a Fixed Grid: Discretize the continuous space of latent ability $\theta$ into a fine, equally spaced vector of $M$ points spanning a practical range (e.g., from $-4$ to $+4$ to capture most standard normal density):$$\theta = [\theta_1, \theta_2, \dots, \theta_M], \quad \text{with step size } \Delta\theta = \theta_{j+1} - \theta_j$$Initialize Prior Density: Evaluate the standard normal prior distribution over this entire grid to form the initial density vector $\mathbf{f}^{(0)}$:$$f^{(0)}_j = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta_j^2}{2}\right) \quad \text{for } j = 1, 2, \dots, M$$2. Sequential Update LoopFor each step $k$ where a new response $y_k \in \{0, 1\}$ is collected for an item with parameters $a_k$ and $b_k$:Compute Item Probabilities: Calculate the probability of a correct response at every grid node:$$p_k(\theta_j) = \frac{1}{1 + \exp\left(-a_k(\theta_j - b_k)\right)} \quad \text{for } j = 1, 2, \dots, M$$Calculate Likelihood Contribution: Evaluate the likelihood vector $\mathbf{L}_k$ across the grid based on the outcome:$$L_k(y_k \mid \theta_j) = [p_k(\theta_j)]^{y_k} [1 - p_k(\theta_j)]^{1 - y_k}$$Pointwise Multiplication (Unnormalized Update): Compute the unnormalized posterior density vector $\mathbf{\tilde{f}}^{(k)}$ by multiplying the likelihood element-wise with the previous step's normalized posterior vector $\mathbf{f}^{(k-1)}$:$$\tilde{f}^{(k)}_j = L_k(y_k \mid \theta_j) \cdot f^{(k-1)}_j \quad \text{for } j = 1, 2, \dots, M$$3. Sequential Normalization StepTo turn the unnormalized vector into a valid probability density function where $\int f(\theta) d\theta = 1$, you must numerically compute the integral of the unnormalized vector and divide by it.Using the Trapezoidal Rule for higher numerical accuracy on uniform grids, the total area $I$ under the unnormalized curve is:$$I = \Delta\theta \cdot \left( \frac{\tilde{f}^{(k)}_1 + \tilde{f}^{(k)}_M}{2} + \sum_{j=2}^{M-1} \tilde{f}^{(k)}_j \right)$$(Alternatively, for a highly dense grid, a simple Riemann sum approximation $I = \sum_{j=1}^{M} \tilde{f}^{(k)}_j \cdot \Delta\theta$ is computationally faster and sufficient).Finally, compute the normalized running posterior vector $\mathbf{f}^{(k)}$ for the current step by dividing each element by the integrated area:$$f^{(k)}_j = \frac{\tilde{f}^{(k)}_j}{I} \quad \text{for } j = 1, 2, \dots, M$$This vector $\mathbf{f}^{(k)}$ becomes the input prior vector for item $k+1$.

In [2]:
import numpy as np

# Set random seed for reproducibility of simulated responses and item parameters
np.random.seed(42)

# 1. Simulation Setup
theta_true = 0.75
n_items = 20

# Generate random item parameters for the 20 questions
# Discrimination a_k uniformly sampled between 0.5 and 2.0
a_params = np.random.uniform(0.5, 2.0, size=n_items)
# Difficulty b_k uniformly sampled between -2.0 and 2.0
b_params = np.random.uniform(-2.0, 2.0, size=n_items)

# 2. Grid Definition and Initialization
M = 1000  # Number of grid nodes
theta_grid = np.linspace(-4, 4, M)
delta_theta = theta_grid[1] - theta_grid[0]

# Initialize with Standard Normal Prior: N(0, 1)
posterior = (1.0 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * theta_grid**2)

# Normalize the initial prior using the trapezoidal rule
initial_area = delta_theta * (0.5 * (posterior[0] + posterior[-1]) + np.sum(posterior[1:-1]))
posterior /= initial_area

# Arrays to store estimators over the timeline
eap_timeline = []
map_timeline = []
responses = []

print(f"Starting Simulation with True Theta = {theta_true}\n")
print(f"{'Item':<6}{'a_k':<8}{'b_k':<8}{'Resp (y_k)':<12}{'EAP Est':<10}{'MAP Est':<10}")
print("-" * 56)

# 3. Sequential Update Loop
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # Calculate probability of correct response given true ability
    p_true = 1.0 / (1.0 + np.exp(-a_k * (theta_true - b_k)))

    # Simulate user response (1 = correct, 0 = incorrect)
    y_k = np.random.binomial(n=1, p=p_true)
    responses.append(y_k)

    # Compute item probabilities across the entire grid
    p_grid = 1.0 / (1.0 + np.exp(-a_k * (theta_grid - b_k)))

    # Calculate likelihood contribution for each grid node
    likelihood = (p_grid ** y_k) * ((1.0 - p_grid) ** (1 - y_k))

    # Pointwise multiplication (unnormalized update)
    unnormalized_posterior = likelihood * posterior

    # Numerical Normalization via Trapezoidal Rule
    area = delta_theta * (0.5 * (unnormalized_posterior[0] + unnormalized_posterior[-1]) + np.sum(unnormalized_posterior[1:-1]))
    posterior = unnormalized_posterior / area

    # 4. Extract Estimators
    # Expected A Posteriori (EAP): Integral of theta * f(theta) d_theta
    eap_integrand = theta_grid * posterior
    eap_est = delta_theta * (0.5 * (eap_integrand[0] + eap_integrand[-1]) + np.sum(eap_integrand[1:-1]))
    eap_timeline.append(eap_est)

    # Maximum A Posteriori (MAP): Theta value that maximizes the posterior density
    map_index = np.argmax(posterior)
    map_est = theta_grid[map_index]
    map_timeline.append(map_est)

    # Print progress tracking row by row
    print(f"{k+1:<6}{a_k:<8.2f}{b_k:<8.2f}{y_k:<12}{eap_est:<10.3f}{map_est:<10.3f}")

print("\nFinal Simulation Summary:")
print(f"True Hidden Ability:  {theta_true}")
print(f"Final EAP Estimate:   {eap_timeline[-1]:.3f}")
print(f"Final MAP Estimate:   {map_timeline[-1]:.3f}")

Starting Simulation with True Theta = 0.75

Item  a_k     b_k     Resp (y_k)  EAP Est   MAP Est   
--------------------------------------------------------
1     1.06    0.45    1           0.515     0.517     
2     1.93    -1.44   1           0.596     0.549     
3     1.60    -0.83   1           0.730     0.645     
4     1.40    -0.53   0           0.146     0.068     
5     0.73    -0.18   1           0.287     0.212     
6     0.73    1.14    1           0.494     0.420     
7     0.59    -1.20   1           0.565     0.492     
8     1.80    0.06    1           0.773     0.693     
9     1.40    0.37    1           0.951     0.877     
10    1.56    -1.81   1           0.961     0.885     
11    0.53    0.43    0           0.856     0.789     
12    1.95    -1.32   1           0.869     0.797     
13    1.75    -1.74   1           0.878     0.805     
14    0.82    1.80    1           1.052     0.981     
15    0.77    1.86    0           0.963     0.901     
16    0.78    1.23 

Q2).

In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Generate an array over the probability domain [0, 1]
theta_vals = np.linspace(0, 1, 500)

# Define the three distinct parameter pairs
states = [
    {"alpha": 1, "beta": 1, "name": "Uninformative: alpha=1, beta=1", "color": "gray"},
    {"alpha": 2, "beta": 8, "name": "Right-skewed: alpha=2, beta=8", "color": "blue"},
    {"alpha": 8, "beta": 2, "name": "Left-skewed: alpha=8, beta=2", "color": "orange"}
]

# Initialize the Plotly figure
fig = go.Figure()

for state in states:
    # Compute the probability density function (PDF)
    pdf_vals = beta.pdf(theta_vals, state["alpha"], state["beta"])

    fig.add_trace(go.Scatter(
        x=theta_vals,
        y=pdf_vals,
        mode='lines',
        name=state["name"],
        line=dict(color=state["color"], width=2.5)
    ))

# Format the chart layout
fig.update_layout(
    title="Beta Distribution Probability Density Functions (PDFs)",
    xaxis_title="Conversion Rate ($\\theta$)",
    yaxis_title="Density",
    template="plotly_white",
    hovermode="x unified",
    xaxis=dict(range=[0, 1])
)

# Render the plot
fig.show()

1. Likelihood Contribution of a Single ResponseA single user interaction $y_k \in \{0, 1\}$ at step $k$ follows a Bernoulli trial where the probability of a click ($y_k = 1$) given the conversion rate $\theta$ is defined as $P(Y_k = 1 \mid \Theta = \theta) = \theta$.We can write the mathematical likelihood contribution $L(y_k \mid \theta)$ of this single isolated response compactly as:$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$2. Joint Likelihood Function for the Running History VectorAssuming each user interaction is conditionally independent given the true conversion rate $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual likelihood contributions up to step $k$:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta) = \prod_{i=1}^{k} \theta^{y_i} (1 - \theta)^{1 - y_i}$$By combining the exponents across the product, the joint likelihood can also be expressed in terms of the total number of clicks gathered up to step $k$:$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{\sum_{i=1}^{k} y_i} (1 - \theta)^{k - \sum_{i=1}^{k} y_i}$$

1. Derivation of the Recursive Algebraic RelationshipBy Bayes' Theorem, the posterior density at step $k$ is proportional to the product of the likelihood contribution of the new observation $y_k$ and the prior distribution at step $k$ (which is the posterior distribution from step $k-1$):$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Substituting the single-item Bernoulli likelihood contribution $L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$, we get the recursive relationship:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{y_k} (1 - \theta)^{1 - y_k} \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$2. Proof of Beta-Binomial ConjugacyAssume the prior state at step $k$ is a Beta distribution with parameters $\alpha_{k-1}$ and $\beta_{k-1}$:$$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) = \frac{1}{\text{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$Substituting this density into the recursive relationship yields:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{y_k} (1 - \theta)^{1 - y_k} \cdot \left[ \frac{1}{\text{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$Dropping the constants that do not depend on $\theta$, we combine the exponential bases:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{\alpha_{k-1} + y_k - 1} (1 - \theta)^{\beta_{k-1} + (1 - y_k) - 1}$$This functional form matches the kernel of a Beta distribution, proving that the posterior distribution stays in the Beta family. The closed-form simple arithmetic updates for the parameters are:$$\alpha_k = \alpha_{k-1} + y_k$$$$\beta_k = \beta_{k-1} + 1 - y_k$$3. Computation of the Posterior MeanThe expected value (mean) of a $\text{Beta}(\alpha_k, \beta_k)$ distribution is given by $\frac{\alpha_k}{\alpha_k + \beta_k}$.Substituting the updated closed-form parameters from step $k$, the posterior mean of the latent parameter $\Theta$ is:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\alpha_{k-1} + y_k}{\alpha_{k-1} + \beta_{k-1} + 1}$$

1. Mathematical Peak Shifting for $y_k = 1$ vs. $y_k = 0$The peak of a $\text{Beta}(\alpha_k, \beta_k)$ distribution is its mode. For a non-trivial Beta distribution ($\alpha_k > 1, \beta_k > 1$), the mode is:$$\text{Mode}(\Theta) = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$Using the closed-form updates $\alpha_k = \alpha_{k-1} + y_k$ and \ \beta_k = \beta_{k-1} + 1 - y_k:Observed Click ($y_k = 1$):The parameters update to $\alpha_k = \alpha_{k-1} + 1$ and $\beta_k = \beta_{k-1}$. The new peak location becomes:$$\text{Mode}_{y_k=1} = \frac{\alpha_{k-1}}{\alpha_{k-1} + \beta_{k-1} - 1}$$Comparing this to the previous mode $\frac{\alpha_{k-1} - 1}{\alpha_{k-1} + \beta_{k-1} - 2}$, the numerator increases relative to the denominator, shifting the peak to the right toward $1$.Observed Non-Click ($y_k = 0$):The parameters update to $\alpha_k = \alpha_{k-1}$ and $\beta_k = \beta_{k-1} + 1$. The new peak location becomes:$$\text{Mode}_{y_k=0} = \frac{\alpha_{k-1} - 1}{\alpha_{k-1} + \beta_{k-1} - 1}$$The denominator increases while the numerator stays constant, shifting the peak to the left toward $0$.

The exact closed-form equations used to evaluate the point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$ are:Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$):$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$):$$\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$(Note: The MAP formula is valid for $\alpha_k > 1$ and $\beta_k > 1$.)

In [4]:
import numpy as np
import plotly.graph_objects as go

# Set seed for reproducible user interactions
np.random.seed(42)

# 1. Simulation Setup
theta_true = 0.35
n_impressions = 100

# 2. Initialize State
alpha = 1
beta = 1

# Lists to track history across steps (starting from step 0 prior state)
# Mode/MAP for alpha=1, beta=1 is undefined/uniform, so we initialize step 0 at 0.5
steps = [0]
bayes_estimates = [alpha / (alpha + beta)]
map_estimates = [0.5]

# 3. Track Estimators Loop
for k in range(1, n_impressions + 1):
    # Simulate Responses: Draw from U(0,1), response is 1 if draw < theta_true
    random_draw = np.random.uniform(0, 1)
    y_k = 1 if random_draw < theta_true else 0

    # Analytical Closed-Form Update
    alpha += y_k
    beta += (1 - y_k)

    # Compute Point Estimators
    theta_bayes = alpha / (alpha + beta)

    # Ensure MAP is protected if alpha or beta equals 1, otherwise use standard formula
    if alpha > 1 and beta > 1:
        theta_map = (alpha - 1) / (alpha + beta - 2)
    else:
        theta_map = 0.5

    # Append to timelines
    steps.append(k)
    bayes_estimates.append(theta_bayes)
    map_estimates.append(theta_map)

# 4. Visualize with Plotly
fig = go.Figure()

# Add Running Posterior Mean (Bayes) line
fig.add_trace(go.Scatter(
    x=steps, y=bayes_estimates,
    mode='lines', name='Posterior Mean (Bayes)',
    line=dict(color='blue', width=2)
))

# Add Running Maximum A Posteriori (MAP) line
fig.add_trace(go.Scatter(
    x=steps, y=map_estimates,
    mode='lines', name='Maximum A Posteriori (MAP)',
    line=dict(color='orange', width=2, dash='dash')
))

# Add static horizontal reference line for true value
fig.add_shape(
    type="line", x0=0, x1=n_impressions, y0=theta_true, y1=theta_true,
    line=dict(color="red", width=2, dash="dot"),
)
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='red', width=2, dash='dot'),
    name=f'True CTR (\u03b8_true = {theta_true})'
))

# Update layout details
fig.update_layout(
    title="Sequential Bayesian CTR Tracking Over 100 Impressions",
    xaxis_title="Impression Step (k)",
    yaxis_title="Estimated Click-Through Rate",
    template="plotly_white",
    hovermode="x unified",
    yaxis=dict(range=[0, 1])
)

# Render the plot
fig.show()

Q3).

In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Generate an array over the restricted domain [0.01, 1.0]
theta_vals = np.linspace(0.01, 1.0, 500)

# Define the prior Beta parameters
alpha_0 = 8
beta_0 = 1.5

# Compute the probability density function (PDF)
pdf_vals = beta.pdf(theta_vals, alpha_0, beta_0)

# Initialize the Plotly figure
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=theta_vals,
    y=pdf_vals,
    mode='lines',
    name='Initial Prior Beta(8, 1.5)',
    line=dict(color='blue', width=2.5)
))

# Format the chart layout
fig.update_layout(
    title="Initial Prior Density Function: Beta(8, 1.5)",
    xaxis_title="Remaining Stiffness Efficiency Factor ($\\theta$)",
    yaxis_title="Density",
    template="plotly_white",
    hovermode="x unified",
    xaxis=dict(range=[0, 1])
)

# Render the plot
fig.show()

Analytical Calculation of $\mathbb{E}[\Theta^{(0)}]$The expected value (mean) of a standard Beta distribution parameterized by $\alpha$ and $\beta$ is computed as:$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha_0}{\alpha_0 + \beta_0}$$Substituting the given shape parameters $\alpha_0 = 8$ and $\beta_0 = 1.5$:$$\mathbb{E}[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} = \frac{16}{19} \approx 0.8421$$

1. Likelihood Contribution of a Single ResponseThe degradation model states that a noisy measurement $y_k$ is given by:$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \quad \text{where } \epsilon_k \sim \mathcal{N}(0, \sigma^2)$$Taking the natural logarithm of both sides transforms the relationship into a linear form:$$\log y_k = \log(\theta \cdot K_{\text{nominal}}) + \epsilon_k$$Since $\epsilon_k$ is normally distributed with a mean of $0$ and variance of $\sigma^2$, the log-transformed observation follows a normal distribution:$$\log y_k \sim \mathcal{N}\left(\log(\theta \cdot K_{\text{nominal}}), \, \sigma^2\right)$$Using the transformation properties of a log-normal distribution, the probability density function for the untransformed measurement $y_k$ determines its single-item likelihood contribution $L(y_k \mid \theta)$:$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left[\log y_k - \log(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right)$$2. Joint Likelihood Function for the Running History VectorAssuming that the measurement noise processes $\epsilon_i$ are independent across successive inspection increments, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual log-normal likelihood components up to step $k$:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta) = \prod_{i=1}^{k} \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{\left[\log y_i - \log(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right)$$Factoring out the constants and converting the product of exponentials into a summation within the exponent yields:$$L(\mathbf{y}^{(k)} \mid \theta) = \left(\frac{1}{\sigma \sqrt{2\pi}}\right)^k \left( \prod_{i=1}^{k} \frac{1}{y_i} \right) \exp\left( -\sum_{i=1}^{k} \frac{\left[\log y_i - \log(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right)$$

Why an Exact Closed-Form Analytical Solution Does Not ExistAn exact closed-form analytical solution does not exist because the Beta prior and the log-normal structural likelihood are non-conjugate.Incompatible Functional Kernels: The kernel of a Beta distribution is polynomial in $\theta$ and $(1-\theta)$, following the form $\theta^{\alpha-1}(1-\theta)^{\beta-1}$. In contrast, the log-normal likelihood contribution contains the parameter inside a squared logarithm within an exponential function: $\exp\left(-\frac{(\log y_k - \log \theta - \log K_{\text{nominal}})^2}{2\sigma^2}\right)$. Multiplying these kernels yields a mathematical expression that does not match any standard, known probability distribution family.Intractable Normalizing Integral: To obtain the exact normalized posterior density, you must calculate the denominator of Bayes' Theorem by integrating the product of the prior and likelihood over the domain $(0, 1]$:$$\int_{0}^{1} \theta^{\alpha-1}(1-\theta)^{\beta-1} \exp\left(-\frac{\left[\log y_k - \log(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2}\right) d\theta$$Because of the transcendental coupling between the polynomial terms and the squared logarithm inside the exponent, this integral cannot be evaluated analytically using elementary or standard special functions. Thus, numerical methods (like a fixed grid) are strictly required.Recursive Relationship for the Posterior DensityBy Bayes' Theorem, the posterior density at step $k$ is proportional to the product of the single-item likelihood contribution at that step and the prior density (which is the posterior distribution from step $k-1$).Dropping all multiplicative constants that do not depend on the parameter $\theta$ (such as $y_k$, $\sigma$, and $\sqrt{2\pi}$), the recursive relationship up to a proportionality constant is:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \exp\left( -\frac{\left[\log y_k - \log(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

1. The Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$)The posterior mean (or Expected A Posteriori estimate) is the expected value of the parameter $\theta$ calculated with respect to the normalized posterior distribution. Over the bounded domain $(0, 1]$, it is defined by the following definite integral equation:$$\hat{\theta}_{\text{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$If expressed explicitly in terms of the unnormalized posterior density (the product of the likelihood and the prior), it matches the ratio of two definite integrals:$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\int_{0}^{1} \theta \cdot \exp\left( -\frac{\left[\log y_k - \log(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \, d\theta}{\int_{0}^{1} \exp\left( -\frac{\left[\log y_k - \log(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \, d\theta}$$2. The Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$)The MAP estimate is the mode of the posterior distribution, representing the parameter value that maximizes the density function. Rather than using an area integration, it is defined using an optimization equation over the bounded interval:$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$Since the normalizing constant in the denominator does not change the location of the maximum, it can be evaluated directly from the unnormalized components:$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} \left[ \exp\left( -\frac{\left[\log y_k - \log(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \right]$$

1. Grid Definition and Boundary HandlingTo handle the physical boundary limits $\theta \in (0, 1]$ without encountering computational singularities—specifically the $\log(0)$ term in the log-normal likelihood—discretize the parameter space using a fine, equally spaced grid of $M$ nodes starting slightly above zero:$$\theta = [\theta_1, \theta_2, \dots, \theta_M]$$Lower Bound ($\theta_1$): Set to a small positive value near zero (e.g., $\theta_1 = 0.001$) to prevent numerical errors during log-transformation.Upper Bound ($\theta_M$): Set exactly to the pristine limit ($\theta_M = 1.0$).Grid Spacing ($\Delta\theta$): Computed uniformly as $\Delta\theta = \frac{\theta_M - \theta_1}{M - 1}$.2. Prior InitializationEvaluate the initial prior density function $\text{Beta}(\alpha_0=8, \beta_0=1.5)$ at every node on the grid to construct the initial probability density vector $\mathbf{f}^{(0)}$:$$f^{(0)}_j = \frac{1}{\text{B}(8, 1.5)} \theta_j^{8 - 1} (1 - \theta_j)^{1.5 - 1} \quad \text{for } j = 1, 2, \dots, M$$Normalize this vector using the trapezoidal rule (detailed in Step 4) to ensure the initial total area under the discrete curve equals exactly $1.0$.3. Sequential Likelihood UpdateWhen a new continuous sensor measurement $y_k$ is observed at inspection milestone $k$:Evaluate Pointwise Likelihood: Compute the log-normal structural likelihood vector $\mathbf{L}_k$ across each grid node $\theta_j$:$$L_k(y_k \mid \theta_j) = \exp\left( -\frac{\left[\log y_k - \log(\theta_j \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right) \quad \text{for } j = 1, 2, \dots, M$$Compute Unnormalized Posterior: Perform element-wise multiplication between the new likelihood vector and the normalized posterior vector from the previous step ($\mathbf{f}^{(k-1)}$):$$\tilde{f}^{(k)}_j = L_k(y_k \mid \theta_j) \cdot f^{(k-1)}_j \quad \text{for } j = 1, 2, \dots, M$$4. Sequential Normalization via the Trapezoidal RuleTo convert the unnormalized vector $\mathbf{\tilde{f}}^{(k)}$ back into a valid probability density function, numerically approximate the total area $I$ under the grid using the composite trapezoidal rule:$$I = \Delta\theta \cdot \left( \frac{\tilde{f}^{(k)}_1 + \tilde{f}^{(k)}_M}{2} + \sum_{j=2}^{M-1} \tilde{f}^{(k)}_j \right)$$Calculate the fully updated, normalized running posterior density vector $\mathbf{f}^{(k)}$ by scaling each coordinate by the calculated area:$$f^{(k)}_j = \frac{\tilde{f}^{(k)}_j}{I} \quad \text{for } j = 1, 2, \dots, M$$This vector $\mathbf{f}^{(k)}$ represents the current structural health state and is passed directly forward to serve as the baseline prior for the next incoming inspection measurement.

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Set random seed for reproducible sensor streams
np.random.seed(42)

# 1. Simulation Constants
theta_true = 0.68
K_nominal = 50.0  # kN/mm
sigma = 0.15
n_measurements = 15

# 2. Grid Setup (Handling the physical boundaries safely)
M = 1000
theta_grid = np.linspace(0.001, 1.0, M)
delta_theta = theta_grid[1] - theta_grid[0]

# 3. Prior Initialization: Beta(8, 1.5) representing the optimistic healthy state
alpha_0, beta_0 = 8, 1.5
posterior = beta.pdf(theta_grid, alpha_0, beta_0)
# Ensure perfect normalization using the trapezoidal rule
posterior /= np.trapezoid(posterior, dx=delta_theta)

# Timeline tracking arrays
steps = [0]
bayes_estimates = [np.trapezoid(theta_grid * posterior, dx=delta_theta)]
map_estimates = [theta_grid[np.argmax(posterior)]]

# Structure to save full distributions at specified milestones
milestones = {0, 1, 2, 5, 10, 15}
saved_posteriors = {0: posterior.copy()}

# 4. Simulation and Update Loop
for k in range(1, n_measurements + 1):
    # Simulate Sensor Stream: Generate log-normal reading centered at theta_true
    epsilon_k = np.random.normal(0, sigma)
    y_k = theta_true * K_nominal * np.exp(epsilon_k)

    # Evaluate Structural Log-Normal Likelihood across the grid
    log_lik_term = -((np.log(y_k) - np.log(theta_grid * K_nominal)) ** 2) / (2 * sigma ** 2)
    # Exponentiate to get the raw likelihood shape (dropping global coefficients)
    likelihood = np.exp(log_lik_term)

    # Pointwise multiplication with the previous step's posterior
    unnormalized_posterior = likelihood * posterior

    # Sequential Normalization using np.trapezoid
    area = np.trapezoid(unnormalized_posterior, dx=delta_theta)
    posterior = unnormalized_posterior / area

    # Compute and Store Running Point Estimators
    eap_est = np.trapezoid(theta_grid * posterior, dx=delta_theta)
    map_est = theta_grid[np.argmax(posterior)]

    steps.append(k)
    bayes_estimates.append(eap_est)
    map_estimates.append(map_est)

    # Save snapshots for specified milestones
    if k in milestones:
        saved_posteriors[k] = posterior.copy()

# 5. Visualization 1: Progression of Full Posterior Density Curves
fig1 = go.Figure()
colors = {0: 'gray', 1: 'lightblue', 2: 'cyan', 5: 'blue', 10: 'purple', 15: 'red'}

for m in sorted(milestones):
    fig1.add_trace(go.Scatter(
        x=theta_grid, y=saved_posteriors[m],
        mode='lines', name=f'Step k = {m}',
        line=dict(color=colors[m], width=2 if m != 15 else 3)
    ))

fig1.update_layout(
    title="Progression of Bounded Posterior Densities Over Time",
    xaxis_title="Stiffness Efficiency Factor (\\theta)",
    yaxis_title="Probability Density",
    template="plotly_white",
    xaxis=dict(range=[0, 1])
)
fig1.show()

# 6. Visualization 2: Convergence Tracking Timeline
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=steps, y=bayes_estimates,
    mode='lines+markers', name='Posterior Mean (Bayes)',
    line=dict(color='blue', width=2)
))

fig2.add_trace(go.Scatter(
    x=steps, y=map_estimates,
    mode='lines+markers', name='Maximum A Posteriori (MAP)',
    line=dict(color='orange', width=2, dash='dash')
))

# Horizontal true value benchmark
fig2.add_shape(
    type="line", x0=0, x1=n_measurements, y0=theta_true, y1=theta_true,
    line=dict(color="red", width=2, dash="dot"),
)
fig2.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='red', width=2, dash='dot'),
    name=f'True Stiffness (\\theta_{{true}} = {theta_true})'
))

fig2.update_layout(
    title="Convergence Timeline of Point Estimators",
    xaxis_title="Inspection Step (k)",
    yaxis_title="Estimated Stiffness Efficiency",
    template="plotly_white",
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    yaxis=dict(range=[0.5, 1.0])
)
fig2.show()

Q4).

We want to find the conditional probability that the latent variable $C_i$ equals $k$, given that we have observed the data point $x_i$. By Bayes' theorem:$$P(C_i = k \mid X_i = x_i) = \frac{p(x_i \mid C_i = k) \cdot P(C_i = k)}{p(x_i)}$$Step 2: Substitute the Prior and Conditional DensityFrom the model specifications:The prior probability of selecting cluster $k$ is $P(C_i = k) = \phi_k$.The conditional density of the observation given cluster $k$ is the multivariate Gaussian density $p(x_i \mid C_i = k) = \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$.Substituting these into the numerator gives:$$P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{p(x_i)}$$Step 3: Substitute the Marginal Density (Denominator)As derived in the previous task using the law of total probability, the marginal density in the denominator is the sum of the joint probabilities across all $K$ possible clusters:$$p(x_i) = \sum_{j=1}^{K} \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)$$Plugging this expression into the denominator yields the final updating equation for the responsibilities:$$\gamma_{ik} = P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$Expanded Algebraic Form to see the exact mathematical operations performed on a $d$-dimensional vector $x_i$, we can expand the multivariate Gaussian density function:$$\mathcal{N}(x_i \mid \mu_k, \Sigma_k) = \frac{1}{(2\pi)^{d/2} \vert{}\Sigma_k\vert{}^{1/2}} \exp\left( -\frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right)$$Substituting this expansion into our conditional update equation gives the full, detailed expression:$$\gamma_{ik} = \frac{\frac{\phi_k}{(2\pi)^{d/2} \vert{}\Sigma_k\vert{}^{1/2}} \exp\left( -\frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right)}{\sum_{j=1}^{K} \frac{\phi_j}{(2\pi)^{d/2} \vert{}\Sigma_j\vert{}^{1/2}} \exp\left( -\frac{1}{2} (x_i - \mu_j)^T \Sigma_j^{-1} (x_i - \mu_j) \right)}$$We can cancel out the constant $(2\pi)^{d/2}$ from both the numerator and the denominator, resulting in the final simplified step:$$\gamma_{ik} = \frac{\phi_k \vert{}\Sigma_k\vert{}^{-1/2} \exp\left( -\frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right)}{\sum_{j=1}^{K} \phi_j \vert{}\Sigma_j\vert{}^{-1/2} \exp\left( -\frac{1}{2} (x_i - \mu_j)^T \Sigma_j^{-1} (x_i - \mu_j) \right)}$$

Why $\gamma_{ik}$ is Interpreted as a Posterior Probability of Cluster MembershipThe term $\gamma_{ik}$ is interpreted as a posterior probability because it updates our belief about cluster assignment after observing empirical data:Prior vs. Posterior Knowledge: Before looking at the specific data point $x_i$, our baseline belief that the point belongs to cluster $k$ is given entirely by the prior probability $\phi_k$. Once we observe its physical features or location $x_i$, we calculate the likelihood $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ that cluster $k$ could have generated it.Bayesian Updating: The calculation directly applies Bayes' rule to combine these two elements:$$\text{Posterior} = \frac{\text{Likelihood} \times \text{Prior}}{\text{Evidence}}$$Proper Probability Properties: The denominator sums the joint probabilities across all $K$ groups, acting as a normalizer. This bounds the quantity strictly between 0 and 1 ($\gamma_{ik} \in [0, 1]$) and guarantees that the responsibilities for a single data point sum up to exactly one across all possible clusters ($\sum_{k=1}^K \gamma_{ik} = 1$).Consequently, $\gamma_{ik}$ represents the updated, conditional probability that data point $x_i$ belongs to cluster $k$ given its observed spatial profile.

Proof that $\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i)$By the mathematical definition of conditional expectation for a discrete random variable, the expected value is the sum of all possible values the variable can take, each multiplied by its corresponding conditional probability.The indicator variable $Z_{ik}$ can only take two possible values: $1$ or $0$. Therefore, we write the conditional expectation as:$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = \sum_{z \in \{0, 1\}} z \cdot P(Z_{ik} = z \mid X_i = x_i)$$Expanding this sum explicitly for its two possible states gives:$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = \left( 1 \cdot P(Z_{ik} = 1 \mid X_i = x_i) \right) + \left( 0 \cdot P(Z_{ik} = 0 \mid X_i = x_i) \right)$$Simplifying the arithmetic drops the second term completely:$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(Z_{ik} = 1 \mid X_i = x_i)$$According to the definition of the one-hot encoded vector, the event $Z_{ik} = 1$ occurs if and only if the data point belongs to cluster $k$ ($C_i = k$). Substituting this equivalence yields the required result:$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i)$$2. Proof that $\mathbb{E}[Z_i \mid X_i = x_i] = [\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$By the linearity property of the expectation operator, taking the expected value of a random vector is equivalent to taking the expected value of each of its individual component scalar elements:$$\mathbb{E}[Z_i \mid X_i = x_i] = \mathbb{E} \left[ \begin{bmatrix} Z_{i1} \\ Z_{i2} \\ \vdots \\ Z_{iK} \end{bmatrix} \;\middle\vert{}\; X_i = x_i \right] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i = x_i] \\ \mathbb{E}[Z_{i2} \mid X_i = x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i = x_i] \end{bmatrix}$$Using the identity proved in Step 1, we replace each element $\mathbb{E}[Z_{ik} \mid X_i = x_i]$ with its conditional cluster probability $P(C_i = k \mid X_i = x_i)$:$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} P(C_i = 1 \mid X_i = x_i) \\ P(C_i = 2 \mid X_i = x_i) \\ \vdots \\ P(C_i = K \mid X_i = x_i) \end{bmatrix}$$From the definition of responsibilities in a Gaussian mixture model, the posterior cluster membership probability for a given component is denoted by $\gamma_{ik} = P(C_i = k \mid X_i = x_i)$. Substituting these terms gives the final vector form:$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

 Mathematical Steps to Formulate Hard Cluster AssignmentTo transition from a probabilistic soft assignment vector to a single deterministic label $\widehat{C}_i$, follow these mathematical steps:Step 1: Write out the soft assignment vectorThe soft assignment vector is given by the conditional expectation of the one-hot encoded latent vector $Z_i$:$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$where each component $\gamma_{ik}$ represents the individual posterior probability that data point $x_i$ belongs to cluster $k$:$$\gamma_{ik} = P(C_i = k \mid X_i = x_i)$$Step 2: Define a decision rule to maximize posterior probabilityTo map this continuous vector of probabilities to a single discrete cluster integer label, we select the cluster index $k \in \{1, 2, \dots, K\}$ that maximizes the conditional probability. This is known as a Maximum A Posteriori (MAP) decision rule:$$\widehat{C}_i = \arg\max_{1 \leq k \leq K} P(C_i = k \mid X_i = x_i)$$Step 3: Substitute the responsibility notationReplacing the conditional probability notation $P(C_i = k \mid X_i = x_i)$ with its defined symbol $\gamma_{ik}$ yields the exact hard assignment formula shown in the image:$$\widehat{C}_i = \arg\max_{1 \leq k \leq K} \gamma_{ik}$$2. Difference Between Soft and Hard Clustering in This ContextSoft Clustering: The data point $x_i$ maintains fractional ownership across all $K$ groups simultaneously. It is defined by the full probability vector $\mathbb{E}[Z_i \mid X_i = x_i]$, where each entry $\gamma_{ik} \in [0, 1]$ captures the model's exact uncertainty or confidence about that specific group membership. This preserves information for points sitting in overlapping regions or boundaries between clusters.Hard Clustering: The data point $x_i$ is rigidly and exclusively assigned to exactly one cluster $\widehat{C}_i$. It collapses the entire probability distribution down to a single index by using a winner-take-all operation ($\arg\max$). Any nuance regarding alternative clusters or uncertainty is completely discarded in favor of a definitive, discrete category label.

 Proof that $\mathbb{E}[X_i \mid C_i = k] = \mu_k$From the model definition, the conditional distribution of an observation $X_i$ given that it belongs to cluster $k$ is a multivariate Gaussian distribution:$$X_i \mid C_i = k \sim \mathcal{N}(\mu_k, \Sigma_k)$$By definition, the expectation of a multivariate normal random variable $\mathbf{Y} \sim \mathcal{N}(\mu, \Sigma)$ is its mean vector $\mu$. Mathematically, this is evaluated using the vector-valued Riemann integral over the data space $\mathbb{R}^d$:$$\mathbb{E}[X_i \mid C_i = k] = \int_{\mathbb{R}^d} x_i \cdot \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i$$Substituting the standard definition of the multivariate Gaussian density yields:$$\mathbb{E}[X_i \mid C_i = k] = \int_{\mathbb{R}^d} x_i \cdot \frac{1}{(2\pi)^{d/2} \vert{}\Sigma_k\vert{}^{1/2}} \exp\left( -\frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right) dx_i$$Letting $z = x_i - \mu_k$ (so $dx_i = dz$), the integral shifts to:$$\mathbb{E}[X_i \mid C_i = k] = \int_{\mathbb{R}^d} (z + \mu_k) \cdot \mathcal{N}(z \mid 0, \Sigma_k) \, dz$$Splitting the integral into two separate terms via linearity gives:$$\mathbb{E}[X_i \mid C_i = k] = \int_{\mathbb{R}^d} z \cdot \mathcal{N}(z \mid 0, \Sigma_k) \, dz + \mu_k \int_{\mathbb{R}^d} \mathcal{N}(z \mid 0, \Sigma_k) \, dz$$The first integral evaluates to $0$ because the integrand $z \cdot \mathcal{N}(z \mid 0, \Sigma_k)$ is an odd function integrated over a symmetric domain centered at zero.The second integral evaluates to $1$ because it integrates a valid probability density function over its entire support.Thus, we obtain:$$\mathbb{E}[X_i \mid C_i = k] = 0 + \mu_k \cdot 1 = \mu_k$$2. Interpretation of $\mu_k$ as the Cluster CenterThe vector $\mu_k$ represents the expected value or average position of all data points generated by cluster $k$. Since a multivariate Gaussian distribution is symmetric and unimodal, its mean matches its mode and median. This makes $\mu_k$ the center of mass and the point of highest point density for that subpopulation in the $d$-dimensional space.

 Why they yield these distinct meanings:$\mathbb{E}[Z_i \mid X_i = x_i]$ gives soft cluster membership because the conditioning is on a fixed observation $x_i$, and the expectation is taken over the cluster indicator variable $Z_i$. This evaluates the system's residual uncertainty about which group generated that specific point, yielding fractional weights across all categories.$\mathbb{E}[X_i \mid C_i = k]$ gives the mean location because it conditions on a specific hidden group $k$ and takes the expectation over the spatial data coordinates $X_i$. This looks inside that cluster to determine its physical center of mass in the feature space


1. Derivation of the Complete-Data Log-LikelihoodThe complete-data likelihood function given in the image is:$$p(x_1, \dots, x_n, z_1, \dots, z_n) = \prod_{i=1}^{n} \prod_{k=1}^{K} \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$To find the complete-data log-likelihood $\ell_c$, we take the natural logarithm ($\log$) of both sides:$$\ell_c = \log \left( \prod_{i=1}^{n} \prod_{k=1}^{K} \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$Using the logarithmic property $\log(\prod a_j) = \sum \log(a_j)$, the products convert into summations:$$\ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} \log \left( \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$Next, using the power rule property $\log(a^b) = b \log(a)$, we bring the exponent $z_{ik}$ out to the front:$$\ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} z_{ik} \log \left( \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right)$$Finally, using the product rule $\log(ab) = \log a + \log b$ inside the parentheses gives the desired expression:$$\ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} z_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$2. Why This Expression is Easy to Maximize if $z_{ik}$ Were KnownIf the latent cluster assignments $z_{ik}$ were known numbers ($0$ or $1$) rather than hidden variables, optimizing the objective function splits into independent sub-problems:Decoupled Parameters: The parameters of interest ($\mu_k, \Sigma_k$) for each cluster $k$ become mathematically decoupled from one another. Because $z_{ik} = 1$ only when point $i$ belongs to cluster $k$, the double summation allows us to optimize each cluster $k$ independently using only the subset of data points assigned to it.No Log-of-Sums: Unlike the incomplete marginal log-likelihood $\sum_i \log (\sum_k \phi_k \mathcal{N}_k)$, taking the logarithm here occurs before any summation. This avoids having a sum inside the log, which would normally bind the parameters together in an intractable non-linear way.Closed-Form Solutions: Maximizing this expression with respect to $\mu_k$ and $\Sigma_k$ reduces directly to standard Standard Maximum Likelihood Estimation (MLE) for a single independent Gaussian distribution. The optimal solutions can be calculated instantly via standard analytical formulas:$$\mu_k = \frac{\sum_{i=1}^n z_{ik}x_i}{\sum_{i=1}^n z_{ik}}, \quad \Sigma_k = \frac{\sum_{i=1}^n z_{ik}(x_i - \mu_k)(x_i - \mu_k)^T}{\sum_{i=1}^n z_{ik}}$$

 The Expected Complete-Data Log-Likelihood Function ($Q$)The E-step computes the expected value of the complete-data log-likelihood function with respect to the conditional distribution of the latent variables, given the observed data $\mathbf{X}$ and the current parameter estimates.By substituting the unknown hard-assignment indicators $z_{ik}$ with their conditional expectations $\mathbb{E}[Z_{ik} \mid X_i = x_i] = \gamma_{ik}$, we obtain the $Q$ function:$$Q = \sum_{i=1}^{n} \sum_{k=1}^{K} \gamma_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$2. Why the E-Step is a Conditional Update of Cluster Membership ProbabilitiesThe E-step can be interpreted as a conditional update of cluster membership probabilities due to the following reasons:Dynamic Probability Shift: Before checking the data vector $x_i$, the initial chance that any given point belongs to cluster $k$ is merely the global mixing weight $\phi_k$. The E-step directly evaluates how well the specific spatial position of $x_i$ matches the current geometric properties ($\mu_k, \Sigma_k$) of cluster $k$.Bayesian Integration of Evidence: The calculation combines the prior belief ($\phi_k$) and the localized component density ($\mathcal{N}_k$) via Bayes' theorem, conditioning the calculation entirely on the realized observation $x_i$.Soft Membership Assignment: This process updates the binary, unobserved variables $z_{ik}$ into continuous values $\gamma_{ik} \in [0, 1]$. Because these values sum to one ($\sum_k \gamma_{ik} = 1$), they act as a valid posterior probability distribution that re-assigns membership dynamically at each iteration based on the latest parameter states.

Derivation of the M-Step Parameter UpdatesTo maximize the expected complete-data log-likelihood function $Q$ with respect to the parameters, we expand the Gaussian term inside the objective function:$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k + \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left( -\frac{d}{2}\log(2\pi) - \frac{1}{2}\log\vert{}\Sigma_k\vert{} - \frac{1}{2}(x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right)$$A. Derivation for $\mu_k^{\text{new}}$To maximize $Q$ with respect to a specific cluster mean vector $\mu_k$, we take the partial derivative of $Q$ with respect to $\mu_k$ and set it to zero. Using the matrix calculus identity $\frac{\partial}{\partial \mathbf{v}} (\mathbf{x} - \mathbf{v})^T \mathbf{A} (\mathbf{x} - \mathbf{v}) = -2\mathbf{A}(\mathbf{x} - \mathbf{v})$ for symmetric $\mathbf{A}$:$$\frac{\partial Q}{\partial \mu_k} = \sum_{i=1}^n \gamma_{ik} \left( \Sigma_k^{-1} (x_i - \mu_k) \right) = 0$$Since $\Sigma_k^{-1}$ is a non-singular matrix, we can multiply both sides by $\Sigma_k$ to eliminate it:$$\sum_{i=1}^n \gamma_{ik} (x_i - \mu_k) = 0 \implies \sum_{i=1}^n \gamma_{ik} x_i - \left(\sum_{i=1}^n \gamma_{ik}\right) \mu_k = 0$$Defining the effective number of points assigned to cluster $k$ as $N_k = \sum_{i=1}^n \gamma_{ik}$, we substitute and solve for $\mu_k$:$$N_k \mu_k = \sum_{i=1}^n \gamma_{ik} x_i \implies \mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i$$B. Derivation for $\Sigma_k^{\text{new}}$To maximize $Q$ with respect to the covariance matrix $\Sigma_k$, we isolate the relevant terms and optimize with respect to the precision matrix $\Lambda_k = \Sigma_k^{-1}$ where $\log\vert{}\Sigma_k\vert{} = -\log\vert{}\Lambda_k\vert{}$:$$Q_{\Sigma_k} = \frac{1}{2} \sum_{i=1}^n \gamma_{ik} \left( \log\vert{}\Lambda_k\vert{} - \text{Tr}\left(\Lambda_k (x_i - \mu_k)(x_i - \mu_k)^T\right) \right)$$Taking the derivative with respect to $\Lambda_k$ using the identities $\frac{\partial \log\vert{}\mathbf{A}\vert{}}{\partial \mathbf{A}} = \mathbf{A}^{-1}$ and $\frac{\partial \text{Tr}(\mathbf{A}\mathbf{B})}{\partial \mathbf{A}} = \mathbf{B}^T$:$$\frac{\partial Q_{\Sigma_k}}{\partial \Lambda_k} = \frac{1}{2} \sum_{i=1}^n \gamma_{ik} \left( \Sigma_k - (x_i - \mu_k)(x_i - \mu_k)^T \right) = 0$$$$\left(\sum_{i=1}^n \gamma_{ik}\right) \Sigma_k = \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k)(x_i - \mu_k)^T$$Substituting $N_k = \sum_{i=1}^n \gamma_{ik}$ and evaluating at the newly computed $\mu_k^{\text{new}}$ yields:$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$C. Derivation for $\phi_k^{\text{new}}$To maximize $Q$ with respect to the mixing weights $\phi_k$, we must respect the equality constraint $\sum_{k=1}^K \phi_k = 1$ using a Lagrange multiplier $\lambda$:$$\mathcal{L}(\phi, \lambda) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k - \lambda \left( \sum_{k=1}^K \phi_k - 1 \right)$$Taking the partial derivative with respect to $\phi_k$ and setting it to zero:$$\frac{\partial \mathcal{L}}{\partial \phi_k} = \sum_{i=1}^n \frac{\gamma_{ik}}{\phi_k} - \lambda = 0 \implies \phi_k = \frac{\sum_{i=1}^n \gamma_{ik}}{\lambda} = \frac{N_k}{\lambda}$$To solve for $\lambda$, we sum both sides over all $K$ clusters:$$\sum_{k=1}^K \phi_k = \sum_{k=1}^K \frac{N_k}{\lambda} \implies 1 = \frac{1}{\lambda} \sum_{k=1}^K \sum_{i=1}^n \gamma_{ik}$$Since the responsibilities for any single data point sum to one ($\sum_{k=1}^K \gamma_{ik} = 1$), the double sum simplifies directly to the total number of data points $n$:$$1 = \frac{n}{\lambda} \implies \lambda = n$$Substituting $\lambda = n$ back into the expression for $\phi_k$ yields:$$\phi_k^{\text{new}} = \frac{N_k}{n}$$2. Physical Interpretation of $\gamma_{ik}$ as a Fractional Membership WeightThe responsibility $\gamma_{ik}$ acts as a continuous, fractional membership weight that determines exactly how much influence data point $x_i$ exerts when re-estimating the parameters of cluster $k$:Weighted Contributions: In the update equations, $\gamma_{ik}$ directly scales each data point's contribution. If a point has $\gamma_{ik} = 0.8$ for cluster $1$ and $\gamma_{ik} = 0.2$ for cluster $2$, $80\%$ of its coordinate values go toward computing the new mean and covariance of cluster $1$, while the remaining $20\%$ goes to cluster $2$.Soft Sample Counting: Instead of counting data points as binary integers, $N_k = \sum_{i=1}^n \gamma_{ik}$ aggregates these fractions to find the effective, continuous sample size belonging to cluster $k$.Preserving Distribution Nuance: This fractional weighting allows data points located in overlapping boundary regions to actively shape the parameters of multiple clusters simultaneously, capturing structural uncertainty rather than introducing artificial boundaries.

Gaussian Mixture Model (GMM) clustering operates as a repeated cycle of conditional updating by alternating between evaluating data compatibility and refining cluster properties. In the E-step, the mixture weight $\phi_k$ serves as the prior probability of a data point belonging to cluster $k$. We then evaluate the Gaussian density $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$, which measures how compatible the observed location $x_i$ is with that specific cluster. By combining this spatial evidence with our prior via Bayes' rule, we compute the responsibility $\gamma_{ik}$, representing the updated posterior probability of cluster $k$ after observing $x_i$. Aggregating these responsibilities across all groups forms the soft assignment vector $\mathbb{E}[Z_i \mid X_i = x_i]$. Next, the M-step updates the cluster parameters ($\phi_k, \mu_k, \Sigma_k$) using these posterior membership probabilities as fractional weights to better fit the data distribution. Repeating this two-step loop demonstrates that Gaussian mixture clustering is fundamentally a probabilistic clustering framework built entirely on the conditional expectations of latent cluster membership variables.

In [20]:
import os
import numpy as np
import pandas as pd
import kagglehub
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(n_components=self.n_components, random_state=self.random_state)

        # Data storage fields
        self.X_train = None
        self.X_test = None
        self.train_df = None
        self.test_df = None

    def prepare_data(self, df, feature1='PURCHASES', feature2='CREDIT_LIMIT'):
        """Cleans, standardizes, and splits the dataset into 80/20 train/test sets."""
        # Drop missing values in selected features
        data_clean = df[[feature1, feature2]].dropna()

        # Extract features matrix
        X = data_clean.values

        # Train-test split (80% training, 20% validation/test)
        X_train_raw, X_test_raw = train_test_split(
            X, test_size=0.20, random_state=self.random_state
        )

        # Standardize features based on training data scaling parameters
        self.X_train = self.scaler.fit_transform(X_train_raw)
        self.X_test = self.scaler.transform(X_test_raw)

        # Save to internal dataframes for easier plotly visualization
        self.train_df = pd.DataFrame(self.X_train, columns=[feature1, feature2])
        self.test_df = pd.DataFrame(self.X_test, columns=[feature1, feature2])
        print(f"Data prepared successfully. Train size: {self.X_train.shape[0]}, Test size: {self.X_test.shape[0]}")

    def fit_model(self):
        """Fits the Gaussian Mixture Model via EM and prints convergence information."""
        if self.X_train is None:
            raise ValueError("Data not prepared. Call prepare_data first.")

        self.gmm.fit(self.X_train)

        # Add predicted hard cluster labels and responsibilities to the train dataframe
        self.train_df['Cluster'] = self.gmm.predict(self.X_train).astype(str)

        print("--- EM Execution Summary ---")
        print(f"Model converged: {self.gmm.converged_}")
        print(f"Iterations required: {self.gmm.n_iter_}")

    def evaluate_test_performance(self):
        """Computes and returns the average log-likelihood score over unseen test data."""
        if self.X_test is None:
            raise ValueError("Data not prepared. Call prepare_data first.")

        avg_log_likelihood = self.gmm.score(self.X_test)
        print("--- Out-of-Sample Performance ---")
        print(f"Average Test Log-Likelihood Score: {avg_log_likelihood:.4f}")

        # Add cluster predictions to test set for plotting
        self.test_df['Cluster'] = self.gmm.predict(self.X_test).astype(str)
        return avg_log_likelihood

    def plot_2d_density_heatmap(self):
        """Generates an empirical 2D Density Heatmap with marginal histograms."""
        fig = px.density_heatmap(
            self.train_df, x=self.train_df.columns[0], y=self.train_df.columns[1],
            marginal_x="histogram", marginal_y="histogram",
            title="Figure 1: Empirical 2D Density Heatmap of Raw Training Data",
            labels={self.train_df.columns[0]: f"Standardized {self.train_df.columns[0]}",
                    self.train_df.columns[1]: f"Standardized {self.train_df.columns[1]}"},
            template="plotly_white"
        )
        fig.show()

    def _generate_contour_grid(self, padding=0.5, resolution=200):
        """Helper method to construct a fine grid mesh and evaluate max responsibilities."""
        x_min, x_max = self.X_train[:, 0].min() - padding, self.X_train[:, 0].max() + padding
        y_min, y_max = self.X_train[:, 1].min() - padding, self.X_train[:, 1].max() + padding

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution), np.linspace(y_min, y_max, resolution))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        # Compute soft assignment matrix (responsibilities gamma_ik)
        responsibilities = self.gmm.predict_proba(grid_points)
        max_responsibilities = np.max(responsibilities, axis=1).reshape(xx.shape)

        return xx[0, :], yy[:, 0], max_responsibilities

    def plot_assignment(self, is_test=False):
        """Generates the contour responsibility map overlaid with data points."""
        df_plot = self.test_df if is_test else self.train_df
        set_title = "Figure 3: Test Assignment Plot" if is_test else "Figure 2: Training Assignment Plot"

        x_range, y_range, z_responsibilities = self._generate_contour_grid()

        fig = go.Figure()

        # 1. Overlay the continuous responsibility background contour
        fig.add_trace(go.Contour(
            x=x_range, y=y_range, z=z_responsibilities,
            colorscale='Viridis',
            contours_coloring='heatmap',
            colorbar=dict(title="Max Posterior Responsibility (\\gamma_{ik})"),
            line_width=0, opacity=0.8,
            hoverinfo='skip'
        ))

        # 2. Add scatter points colored by hard cluster assignment
        for cluster_id in sorted(df_plot['Cluster'].unique()):
            cluster_data = df_plot[df_plot['Cluster'] == cluster_id]
            fig.add_trace(go.Scatter(
                x=cluster_data.iloc[:, 0], y=cluster_data.iloc[:, 1],
                mode='markers', name=f'Cluster {cluster_id}',
                marker=dict(size=5, line=dict(color='white', width=0.5))
            ))

        fig.update_layout(
            title=set_title,
            xaxis_title=f"Standardized {df_plot.columns[0]}",
            yaxis_title=f"Standardized {df_plot.columns[1]}",
            template="plotly_white"
        )
        fig.show()

# --- Interactive Execution Pipeline ---
if __name__ == "__main__":
    # 1. Download latest version of the dataset using kagglehub
    path = kagglehub.dataset_download("arjunbhasin2013/ccdata")
    print("Path to dataset files:", path)

    # 2. Locate and load the CSV file inside the downloaded path
    csv_filename = "CC GENERAL.csv"
    full_csv_path = os.path.join(path, csv_filename)
    df = pd.read_csv(full_csv_path)

    # 3. Initialize pipeline segmenter and execute GMM
    segmenter = GMMFinancialSegmenter(n_components=3)
    segmenter.prepare_data(df, 'PURCHASES', 'CREDIT_LIMIT')
    segmenter.fit_model()
    segmenter.evaluate_test_performance()

    # 4. Generate Interactive Plots
    segmenter.plot_2d_density_heatmap()
    segmenter.plot_assignment(is_test=False) # Train plot
    segmenter.plot_assignment(is_test=True)  # Test plot

Using Colab cache for faster access to the 'ccdata' dataset.
Path to dataset files: /kaggle/input/ccdata
Data prepared successfully. Train size: 7159, Test size: 1790
--- EM Execution Summary ---
Model converged: True
Iterations required: 18
--- Out-of-Sample Performance ---
Average Test Log-Likelihood Score: -1.6890
